# Preparation

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/google/bert/tensorflow2/answer-equivalence-bem/1/saved_model.pb
/kaggle/input/models/google/bert/tensorflow2/answer-equivalence-bem/1/variables/variables.index
/kaggle/input/models/google/bert/tensorflow2/answer-equivalence-bem/1/variables/variables.data-00000-of-00001


In [2]:
!pip install tensorflow-text

In [3]:
# Install Tensorflow libraries first
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text

# Install dependencies
import numpy as np
import scipy
from scipy.special import softmax

2026-05-15 09:38:45.160802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778837925.490740      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778837925.570083      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778837926.327926      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778837926.327991      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778837926.327997      57 computation_placer.cc:177] computation placer alr

# Define settings

In [8]:
vocab_path = 'gs://cloud-tpu-checkpoints/bert/keras_bert/uncased_L-12_H-768_A-12/vocab.txt'

vocab_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.TextFileInitializer(
        filename=vocab_path,
        key_dtype=tf.string,
        key_index=tf.lookup.TextFileIndex.WHOLE_LINE,
        value_dtype=tf.int64,
        value_index=tf.lookup.TextFileIndex.LINE_NUMBER
    ),
    num_oov_buckets=1)

cls_id, sep_id = vocab_table.lookup(tf.convert_to_tensor(['[CLS]', '[SEP]']))

tokenizer = text.BertTokenizer(
    vocab_lookup_table=vocab_table,
    token_out_type=tf.int64,
    preserve_unused_token=True,
    lower_case=True
)

In [9]:
def bertify_example(example):
    question = tokenizer.tokenize(example['question']).merge_dims(1, 2)
    reference = tokenizer.tokenize(example['reference']).merge_dims(1, 2)
    candidate = tokenizer.tokenize(example['candidate']).merge_dims(1, 2)

    input_ids, segment_ids = text.combine_segments(
        (candidate, reference, question),
        cls_id,
        sep_id
    )

    return {'input_ids': input_ids.numpy(), 'segment_ids': segment_ids.numpy()}

def pad(a, length=512):
    return np.append(a, np.zeros(length - a.shape[-1], np.int32))

def bertify_examples(examples):
    input_ids = []
    segment_ids = []
    for example in examples:
        example_inputs = bertify_example(example)
        input_ids.append(pad(example_inputs['input_ids']))
        segment_ids.append(pad(example_inputs['segment_ids']))